In [ ]:
discharge_notes = pd.read_csv(f"{mimic_path}/mimic-iv-note/2.2/note/discharge.csv.gz")
discharge_detail_notes = pd.read_csv(f"{mimic_path}/mimic-iv-note/2.2/note/discharge_detail.csv.gz")
radiology_notes = pd.read_csv(f"{mimic_path}/mimic-iv-note/2.2/note/radiology.csv.gz")
radiology_detail_notes = pd.read_csv(f"{mimic_path}/mimic-iv-note/2.2/note/radiology_detail.csv.gz")

noteevents = pd.concat([discharge_notes, discharge_detail_notes, radiology_notes, radiology_detail_notes])
print("Loading noteevents table completes")

# Filter for Echo notes
# echo_notes = noteevents[noteevents['category'] == 'Echo'].copy()
echo_notes = noteevents[noteevents['text'].str.contains('Echo', case=False, na=False)].copy()

In [ ]:
weight1 = echo_notes["text"].str.extract(r"Weight\s*\(lb\):\s*([^\n\*]*)\n", flags=re.IGNORECASE)[0]
weight2 = echo_notes["text"].str.extract(r"weight\s*[:=]\s*([0-9]+\.?\d*)", flags=re.IGNORECASE)[0]
echo_notes["Weight"] = pd.to_numeric(weight1, errors='coerce')
missing = echo_notes["Weight"].isnull()
echo_notes.loc[missing, "Weight"] = pd.to_numeric(weight2[missing], errors='coerce')
echo_notes.loc[echo_notes["Weight"].astype(str).str.contains('\*', na=False), "Weight"] = None

In [ ]:
import pandas as pd
import re

def extract_echo_fields(df):
    # Indication (greedy up to newline)
    df['Indication'] = df['text'].str.extract(r'Indication:\s*(.*?)\n', flags=re.IGNORECASE)

    # Height: strict SQL, then fallback to flexible (both exclude values containing '*')
    height1 = df["text"].str.extract(r"Height:\s*\(in\)\s*([^\n\*]*)\n", flags=re.IGNORECASE)[0]
    height2 = df["text"].str.extract(r"height\s*[:=]?\s*([0-9]+\.?\d*)", flags=re.IGNORECASE)[0]
    df["Height"] = pd.to_numeric(height1, errors='coerce')
    missing = df["Height"].isnull()
    df.loc[missing, "Height"] = pd.to_numeric(height2[missing], errors='coerce')
    df.loc[df["Height"].astype(str).str.contains('\*', na=False), "Height"] = None

    # Weight: strict SQL, then fallback to flexible


    # BSA: strict SQL, then fallback flexible
    bsa1 = df["text"].str.extract(r"BSA\s*\(m2\):\s*([^\s\*]+)", flags=re.IGNORECASE)[0]
    bsa2 = df["text"].str.extract(r"BSA\s*\(m2\)\s*:?=?\s*([0-9]+\.?\d*)", flags=re.IGNORECASE)[0]
    df["BSA"] = pd.to_numeric(bsa1, errors='coerce')
    missing = df["BSA"].isnull()
    df.loc[missing, "BSA"] = pd.to_numeric(bsa2[missing], errors='coerce')
    df.loc[df["BSA"].astype(str).str.contains('\*', na=False), "BSA"] = None

    # BP: SQL, take full field
    df["BP"] = df["text"].str.extract(r'BP\s*\(mm Hg\):\s*([^\n]*)', flags=re.IGNORECASE)[0]
    # Systolic/diastolic
    systolic = df["text"].str.extract(r'BP\s*\(mm Hg\):\s*([0-9]+)\s*/\s*[0-9]+\s*\n', flags=re.IGNORECASE)[0]
    diastolic = df["text"].str.extract(r'BP\s*\(mm Hg\):\s*[0-9]+\s*/\s*([0-9]+)\s*\n', flags=re.IGNORECASE)[0]
    df["BPSys"] = pd.to_numeric(systolic, errors='coerce')
    df["BPDias"] = pd.to_numeric(diastolic, errors='coerce')

    # HR: SQL, fallback flexible (exclude *)
    hr1 = df["text"].str.extract(r'HR\s*\(bpm\):\s*([^\n\*]*)\n', flags=re.IGNORECASE)[0]
    df["HR"] = pd.to_numeric(hr1, errors='coerce')
    hr2 = df["text"].str.extract(r'HR\s*\(bpm\)?\s*[:=]?\s*([0-9]+\.?\d*)', flags=re.IGNORECASE)[0]
    missing = df["HR"].isnull()
    df.loc[missing, "HR"] = pd.to_numeric(hr2[missing], errors='coerce')
    df.loc[df["HR"].astype(str).str.contains('\*', na=False), "HR"] = None

    # Additional fields, per SQL logic, straight greedy-to-newline:
    for field, key in [("Status", "Status"), ("Test", "Test"), ("Doppler", "Doppler"), ("Contrast", "Contrast"), ("TechnicalQuality", "Technical Quality")]:
        df[field] = df["text"].str.extract(fr"{key}:\s*(.*?)\n", flags=re.IGNORECASE)[0]

    return df


In [ ]:
 echo_notes["Height"] = echo_notes["text"].str.extract(
        r"Height\s*:?\s*\(?in\)?\s*([0-9]+(?:\.\d+)?)", flags=re.IGNORECASE
    ).astype(float)

  # Weight: allow "Weight" plus "(lb)", then number—even with extra spaces or no colon
echo_notes["Weight"] = echo_notes["text"].str.extract(
        r"Weight\s*\(?lb\)?\s*[:=]?\s*([0-9]+(?:\.\d+)?)", flags=re.IGNORECASE
    ).astype(float)

In [ ]:
echo_notes["text"].str.extract(r"Technical Quality:\s*(.*?)(?:\n|$)")

In [ ]:
import pandas as pd
import re

def extract_echo_params(df):
    # Height: can be any occurrence of "Height", optional (in), then number


    # BSA: flexible for "BSA", possibly "m2", and number with decimal
    df["BSA"] = df["text"].str.extract(
        r"BSA\s*\(?m2\)?\s*[:=]?\s*([0-9]+(?:\.\d+)?)", flags=re.IGNORECASE
    ).astype(float)

    # BP: handle "BP" and "mm Hg", then capture full string to newline or comma
    df["BP"] = df["text"].str.extract(
        r"BP\s*\(?mm Hg\)?\s*[:=]?\s*([^\n,]+)", flags=re.IGNORECASE
    )

    # BP systolic: look for fraction style, first group
    df["BPSys"] = df["text"].str.extract(
        r"BP\s*\(?mm Hg\)?\s*[:=]?\s*([0-9]+)\s*/\s*[0-9]+", flags=re.IGNORECASE
    ).astype(float)

    # BP diastolic: second group in fraction
    df["BPDias"] = df["text"].str.extract(
        r"BP\s*\(?mm Hg\)?\s*[:=]?\s*[0-9]+\s*/\s*([0-9]+)", flags=re.IGNORECASE
    ).astype(float)

    # HR: tolerate missing parentheses and various spacings
    df["HR"] = df["text"].str.extract(
        r"HR\s*\(?bpm\)?\s*[:=]?\s*([0-9]+(?:\.\d+)?)(?!%)", flags=re.IGNORECASE
    ).astype(float)

    return df
